# Geo-Nexus / MH-DAPT-CD v3.2 — Phase P2: Colab Preprocessing & Auto-Label Fusion

This notebook executes the full **Phase P2** pipeline as specified in `FINAL_ARCH_3.md` (Part 16, Part 18, Part 22.5, Part 23, Part 24, Part 26.2, Part 27, and Part 29.2):
1. **Environment Setup & Drive Mount** (raw rasters in `/content/drive/MyDrive/geonexus_v3_raw`)
2. **GEE Tile Reassembly** (`load_merged` for multi-tile exports)
3. **17-Channel Tensor Derivation & P0 Pre-Flight Checks** (Harmonization, Quality variance, SAR dB range, and Monsoon $q255$ support)
4. **Single-Source Auto-Label Fusion Import** (Imports from `data.autolabel`)
5. **Zone A AlphaEarth Percentile Tuning Sweep** (`ae_pct_lo` in `[45, 50, 55, 60, 65]` with strict error gating)
6. **Freeze Selected Percentile & Fuse All Zones** (`pune`, `satara`, `vidarbha`)
7. **Spatial AOI Split & 128x128 Tiling** (Stride 64 Train / Stride 128 Test with 128px Hard Buffer)
8. **Normalization Statistics on TRAIN AOI ONLY** (`norm_stats_trainonly.json` — zero test leakage)
9. **Independent 220-Patch Annotation Pool & High-Performance QGIS Export** (Preloaded memory cache, 20 blind, 30 monsoon pairs stratified by cloud quality $q$)
10. **Archive to Drive & Kaggle Dataset Staging Preparation**

## Cell 1: Environment & Google Drive Mount

In [ ]:
# ============ CELL 1: ENVIRONMENT & PATHS ============
!pip -q install rasterio kaggle tqdm pyarrow scipy

from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, glob, shutil, warnings, psutil
import numpy as np
import rasterio
from pathlib import Path
from tqdm.auto import tqdm
warnings.filterwarnings('ignore', category=rasterio.errors.NotGeoreferencedWarning)

# Ensure Python path recognizes /content and local packages
if '/content' not in sys.path:
    sys.path.insert(0, '/content')

# Google Drive & Scratch Directories
DRIVE_RAW  = Path('/content/drive/MyDrive/geonexus_v3_raw')
DRIVE_PROC = Path('/content/drive/MyDrive/geonexus_v3_processed')
LOCAL      = Path('/content/proc')  # Fast local scratch (do heavy writes here)
LOCAL.mkdir(parents=True, exist_ok=True)
DRIVE_PROC.mkdir(parents=True, exist_ok=True)

# Ensure data package directory exists in Colab environment
DATA_DIR = Path('/content/data')
DATA_DIR.mkdir(parents=True, exist_ok=True)
(DATA_DIR / '__init__.py').touch()

# Staging: Copy data modules from Drive/Geo_Watch if present; otherwise write verified code directly
DRIVE_REPO_DATA = Path('/content/drive/MyDrive/Geo_Watch/data')
for mod_name in ['autolabel.py', 'export_qgis.py']:
    src_f = DRIVE_REPO_DATA / mod_name
    dst_f = DATA_DIR / mod_name
    if src_f.exists():
        shutil.copy(src_f, dst_f)
        print(f'Copied {mod_name} from {src_f}')

# Fallback: If autolabel.py is missing, write verified single source of truth
if not (DATA_DIR / 'autolabel.py').exists():
    print('Writing fallback verified autolabel.py into /content/data...')
    (DATA_DIR / 'autolabel.py').write_text('''import numpy as np
from scipy.ndimage import binary_erosion, binary_dilation, binary_opening, label as nd_label

IGNORE = 255
CLASS_NAMES = ['no_change', 'water_gain', 'water_loss', 'construction', 'veg_loss', 'veg_gain', 'other']
EV = {'ob_p2020': 0, 'ob_p2023': 1, 'ob_h2023': 2, 'hansen_ly': 3, 'hansen_tc00': 4, 'dw_built_t1': 5, 'dw_built_t2': 6, 'dw_trees_t1': 7, 'dw_trees_t2': 8, 'jrc_occ': 9}
CFG = dict(ob_hi=0.60, ob_lo=0.20, ob_jump=0.45, tau_ndbi=0.15, ndvi_drop_min=0.05, dw_margin=0.35, tau_ndvi_loss=0.20, ndvi_t1_min=0.40, tau_ndvi_gain=0.20, ndvi_t1_max=0.30, ndvi_t2_min=0.45, mndwi_thresh=0.0, jrc_occ_min=5, jrc_buffer_px=20, ae_pct_hi=92, ae_pct_lo=55, min_blob_px=4)

def load_evidence(zone, root, load_merged_fn):
    e = load_merged_fn(f"{zone}_autolabel_evidence", root).astype(np.float32)
    return {'ob_p2020': e[EV['ob_p2020']]/10000.0, 'ob_p2023': e[EV['ob_p2023']]/10000.0, 'ob_h2023': e[EV['ob_h2023']]/100.0, 'hansen_ly': e[EV['hansen_ly']], 'hansen_tc00': e[EV['hansen_tc00']], 'dw_built_t1': e[EV['dw_built_t1']]/10000.0, 'dw_built_t2': e[EV['dw_built_t2']]/10000.0, 'dw_trees_t1': e[EV['dw_trees_t1']]/10000.0, 'dw_trees_t2': e[EV['dw_trees_t2']]/10000.0, 'jrc_occ': e[EV['jrc_occ']]}

def build_evidence_masks(ev, ndvi1, ndvi2, ndbi1, ndbi2, mndwi1, mndwi2, c=CFG):
    d_ndvi = ndvi2 - ndvi1; d_ndbi = ndbi2 - ndbi1
    e_ob = (ev['ob_p2023'] >= c['ob_hi']) & (ev['ob_p2020'] <= c['ob_lo']) & ((ev['ob_p2023'] - ev['ob_p2020']) >= c['ob_jump'])
    e_ndbi = (d_ndbi >= c['tau_ndbi']) & (d_ndvi <= -c['ndvi_drop_min'])
    e_dw_b = (ev['dw_built_t2'] - ev['dw_built_t1']) >= c['dw_margin']
    construction = e_ob | (e_ndbi & e_dw_b)
    e_hansen = (ev['hansen_ly'] >= 20) & (ev['hansen_ly'] <= 23)
    e_hansen = binary_erosion(e_hansen, np.ones((3, 3)))
    e_ndvi_l = (d_ndvi <= -c['tau_ndvi_loss']) & (ndvi1 >= c['ndvi_t1_min'])
    e_dw_t = (ev['dw_trees_t1'] - ev['dw_trees_t2']) >= c['dw_margin']
    veg_loss = e_hansen | (e_ndvi_l & e_dw_t)
    veg_gain = (d_ndvi >= c['tau_ndvi_gain']) & (ndvi1 <= c['ndvi_t1_max']) & (ndvi2 >= c['ndvi_t2_min'])
    w1, w2 = mndwi1 >= c['mndwi_thresh'], mndwi2 >= c['mndwi_thresh']
    prior = binary_dilation(ev['jrc_occ'] >= c['jrc_occ_min'], np.ones((c['jrc_buffer_px']*2+1,)*2))
    return {'water_gain': (~w1) & w2 & prior, 'water_loss': w1 & (~w2) & prior, 'construction': construction, 'veg_loss': veg_loss, 'veg_gain': veg_gain}

def remove_small_blobs(mask, min_size=4):
    if min_size <= 1: return mask
    labeled, num_features = nd_label(mask)
    if num_features == 0: return mask
    counts = np.bincount(labeled.flat)
    remove = counts < min_size
    remove[0] = False
    out = mask.copy(); out[remove[labeled]] = False; return out

def fuse(masks, ae_score, c=CFG):
    H, W = ae_score.shape
    label = np.zeros((H, W), np.uint8); conf = np.zeros((H, W), np.float32); claimed = np.zeros((H, W), bool)
    for name, cid in [('water_gain', 1), ('water_loss', 2), ('construction', 3), ('veg_loss', 4), ('veg_gain', 5)]:
        m = masks[name] & ~claimed
        m = binary_opening(m, np.ones((3, 3)))
        if c.get('min_blob_px', 0) > 1: m = remove_small_blobs(m, c['min_blob_px'])
        label[m] = cid; claimed |= m
    tau_hi = np.percentile(ae_score, c['ae_pct_hi']); tau_lo = np.percentile(ae_score, c['ae_pct_lo'])
    ae_chg = ae_score >= tau_hi; ae_stab = ae_score <= tau_lo
    conf[claimed & ae_chg] = 1.00; conf[claimed & ~ae_chg & ~ae_stab] = 0.60; label[claimed & ae_stab] = IGNORE
    unclaimed = ~claimed; label[unclaimed & ae_chg] = 6; conf[unclaimed & ae_chg] = 0.50
    label[unclaimed & ae_stab] = 0; conf[unclaimed & ae_stab] = 1.00; label[unclaimed & ~ae_chg & ~ae_stab] = IGNORE
    return label, conf

def report(label, zone):
    tot = label.size; out = {'zone': zone, 'tau_note': 'percentile-based'}
    for i, n in enumerate(CLASS_NAMES): out[n] = round(100.0 * float((label == i).sum()) / tot, 3)
    out['uncertain'] = round(100.0 * float((label == IGNORE).sum()) / tot, 3)
    print(f"[{zone}] " + "  ".join(f"{k}={v}%" for k, v in out.items() if k not in ('zone','tau_note')))
    return out
''')

# Fallback: If export_qgis.py is missing, write verified single source of truth
if not (DATA_DIR / 'export_qgis.py').exists():
    print('Writing fallback verified export_qgis.py into /content/data...')
    qml_payload = '<qgis><pipe><rasterrenderer type="paletted" band="1" opacity="0.5"><colorPalette><paletteEntry value="0" color="#000000" label="no change" alpha="0"/><paletteEntry value="1" color="#1E64DC" label="water gain"/><paletteEntry value="2" color="#50C8DC" label="water loss"/><paletteEntry value="3" color="#FFC800" label="construction"/><paletteEntry value="4" color="#DC3232" label="vegetation loss"/><paletteEntry value="5" color="#32B432" label="vegetation gain"/><paletteEntry value="6" color="#B4B4B4" label="other"/><paletteEntry value="255" color="#FF00FF" label="UNCERTAIN"/></colorPalette></rasterrenderer></pipe></qgis>'
    export_code = (
        'import numpy as np, rasterio\n'
        'from rasterio.windows import Window, transform as win_transform\n'
        'from pathlib import Path\n\n'
        f'QML = """{qml_payload}"""\n\n'
        'def export_patch(zone, r, c, x1, x2, lab, ae, base_tf, crs, out_dir, blind=False, patch=128):\n'
        '    out_dir.mkdir(parents=True, exist_ok=True)\n'
        '    tf = win_transform(Window(c, r, patch, patch), base_tf)\n'
        '    stem = f"{zone}_{r}_{c}"\n'
        '    img = np.stack([x1[2], x1[1], x1[0], x1[6], x2[2], x2[1], x2[0], x2[6]])[:, r:r+patch, c:c+patch]\n'
        '    with rasterio.open(out_dir/f"{stem}.tif", "w", driver="GTiff", height=patch, width=patch, count=8, dtype="float32", crs=crs, transform=tf) as d:\n'
        '        d.write(img.astype(np.float32))\n'
        '        d.descriptions = ("T1_R","T1_G","T1_B","T1_NIR","T2_R","T2_G","T2_B","T2_NIR")\n'
        '    if blind: return\n'
        '    with rasterio.open(out_dir/f"{stem}_ae.tif", "w", driver="GTiff", height=patch, width=patch, count=1, dtype="float32", crs=crs, transform=tf) as d:\n'
        '        d.write(ae[None, r:r+patch, c:c+patch].astype(np.float32))\n'
        '    with rasterio.open(out_dir/f"{stem}_autolabel.tif", "w", driver="GTiff", height=patch, width=patch, count=1, dtype="uint8", crs=crs, transform=tf, nodata=None) as d:\n'
        '        d.write(lab[None, r:r+patch, c:c+patch])\n'
        '    (out_dir/f"{stem}_autolabel.qml").write_text(QML, encoding="utf-8")\n'
    )
    (DATA_DIR / 'export_qgis.py').write_text(export_code)

# Constants strictly matching GEE export definitions
S2_BANDS   = ['B2','B3','B4','B5','B6','B7','B8','B8A','B9','B11','B12']
IDX        = {b: i for i, b in enumerate(S2_BANDS)}  # B4->2, B8->6, B11->9, B3->1
S2_SCALE   = 10000.0       # Reflectance = DN / 10000
SAR_SCALE  = 100.0         # dB = DN / 100
N_TARGET   = 8.0           # Clear observations for quality q = 1.0
PATCH      = 128           # 128x128 patches
STRIDE_TR  = 64            # 50% overlap inside train AOI
STRIDE_TE  = 128           # No overlap in test AOI
BUFFER_PX  = 128           # Hard spatial gap between train and test halves

raw_files = list(DRIVE_RAW.glob('*.tif'))
print(f'Raw TIFF files found in {DRIVE_RAW}: {len(raw_files)}')
assert len(raw_files) > 0, f'No TIFF files found in {DRIVE_RAW}. Check Drive mount or folder path.'

import os, psutil
def show_ram(tag=''):
    proc = psutil.Process(os.getpid())
    vm = psutil.virtual_memory()
    print(f'[{tag}] RSS={proc.memory_info().rss / (1024**3):.2f} GiB | Available={vm.available / (1024**3):.2f} GiB | Total={vm.total / (1024**3):.2f} GiB')

show_ram('Cell 1 Init')




## Cell 2: Reassemble GEE Split Tiles (`load_merged`)

In [ ]:
# ============ CELL 2: TILE MERGING ============
def load_merged(prefix: str, raw_dir: Path = DRIVE_RAW) -> np.ndarray:
    """
    Reassemble a GEE multi-tile Drive export into a single [C, H, W] array.
    GEE names tiles '<prefix>-<ROWOFFSET>-<COLOFFSET>.tif' with zero-padded
    pixel offsets, so the offsets tell us exactly where each tile belongs.
    """
    files = sorted(raw_dir.glob(f'{prefix}*.tif'))
    if not files:
        raise FileNotFoundError(f'No tiles matching prefix "{prefix}" in {raw_dir}')
    if len(files) == 1:
        with rasterio.open(files[0]) as src:
            return src.read()

    tiles = []
    for f in files:
        stem = f.stem
        try:
            row_off, col_off = int(stem.split('-')[-2]), int(stem.split('-')[-1])
        except ValueError:
            raise RuntimeError(f'Unexpected tile filename format: {f.name}')
        with rasterio.open(f) as src:
            tiles.append((row_off, col_off, src.read()))

    C = tiles[0][2].shape[0]
    H = max(r + a.shape[1] for r, _, a in tiles)
    W = max(c + a.shape[2] for _, c, a in tiles)
    out = np.zeros((C, H, W), dtype=tiles[0][2].dtype)
    for r, c, a in tiles:
        out[:, r:r + a.shape[1], c:c + a.shape[2]] = a
    print(f'  [{prefix}] merged {len(files)} tiles -> {out.shape}')
    return out


def load_zone(zone: str, period: str, raw_dir: Path = DRIVE_RAW):
    """Returns (optical[11] reflectance, n_clear[H,W], sar[2] dB)."""
    o   = load_merged(f'{zone}_{period}_optical', raw_dir).astype(np.float32)
    opt, nclear = o[:11] / S2_SCALE, o[11]
    sar = load_merged(f'{zone}_{period}_sar', raw_dir).astype(np.float32) / SAR_SCALE
    # Crop to common extent (GEE tile padding can differ by a few pixels)
    H = min(opt.shape[1], sar.shape[1])
    W = min(opt.shape[2], sar.shape[2])
    return opt[:, :H, :W], nclear[:H, :W], sar[:, :H, :W]

## Cell 3: 17-Channel Derivation, SAR Scaling Fix & P0 Assertions

In [ ]:
# ============ CELL 3: 17 CHANNELS & P0 CHECKS ============
EPS = 1e-6
SAR_CH = [13, 14, 15]  # VV, VH, Cross-Ratio
def derive_17ch(opt, nclear, sar, is_monsoon=False):
    """
    Transforms (opt[11] reflectance, nclear[H,W], sar[2] dB) -> [17, H, W] float32.
    Channels:
      0-10: S2 Reflectance (11 bands)
      11:   NDVI (B8 - B4) / (B8 + B4 + EPS)
      12:   NDBI (B11 - B8) / (B11 + B8 + EPS)
      13:   SAR VV in dB
      14:   SAR VH in dB
      15:   Cross-Ratio (VH - VV in dB)
      16:   Quality mask q:
            - If dry composite: clip(nclear / 8.0, 0, 1)
            - If monsoon single-date: clip(nclear / 255.0, 0, 1) (Section Q9: cs_cdf*255)
    """
    B3, B4, B8, B11 = opt[IDX['B3']], opt[IDX['B4']], opt[IDX['B8']], opt[IDX['B11']]
    ndvi = (B8  - B4 ) / (B8  + B4  + EPS)      # ch 11
    ndbi = (B11 - B8 ) / (B11 + B8  + EPS)      # ch 12
    cr   = sar[1] - sar[0]                      # ch 15 (VH - VV in dB)
    
    if is_monsoon:
        q = np.clip(nclear / 255.0, 0.0, 1.0)   # Section Q9: q = cs_cdf
    else:
        q = np.clip(nclear / N_TARGET, 0.0, 1.0) # n_clear looks / 8
        
    x17 = np.concatenate([
        opt,                     #  0-10 raw reflectance (11 bands)
        ndvi[None], ndbi[None],  # 11-12 optical indices
        sar,                     # 13-14 VV, VH dB
        cr[None],                # 15    cross ratio dB
        q[None],                 # 16    quality mask
    ]).astype(np.float32)
    
    # Defect D4 Fix: Rescale SAR dB by 1/100 so (dB/100)*10000 = dB*100,
    # preventing int16 overflow when converting to int16 shards.
    x17[SAR_CH] = x17[SAR_CH] / 100.0
    return x17
def mndwi(opt):
    """Modified Normalized Difference Water Index (B3 - B11) / (B3 + B11)."""
    B3, B11 = opt[IDX['B3']], opt[IDX['B11']]
    return (B3 - B11) / (B3 + B11 + EPS)
# ---------------- P0 SCIENTIFIC ASSERTIONS ----------------
def p0_check(zone):
    print(f'=== Verifying P0 Assertions for [{zone}] ===')
    o1, n1, s1 = load_zone(zone, 'T1')
    o2, n2, s2 = load_zone(zone, 'T2')
    # 1. Harmonization: median band-wise reflectance difference must be small
    d = np.nanmedian(np.abs(np.nanmedian(o2, axis=(1,2)) - np.nanmedian(o1, axis=(1,2))))
    print(f'  [{zone}] median |d rho| across bands = {d:.4f}')
    assert d < 0.02, f'HARMONIZATION FAILURE in {zone} (d={d:.4f} >= 0.02). Must use S2_SR_HARMONIZED.'
    # 2. Quality mask q variance
    q1 = np.clip(n1 / N_TARGET, 0, 1)
    print(f'  [{zone}] q: mean={q1.mean():.3f} std={q1.std():.3f} min={q1.min():.2f} max={q1.max():.2f}')
    assert q1.std() > 0.02, f'q mask has zero/insufficient variance in {zone} (std={q1.std():.4f} <= 0.02).'
    # 3. SAR sanity check: VV dB over land must be between -30 and +5 dB
    print(f'  [{zone}] VV mean={np.nanmean(s1[0]):.1f} dB, VH mean={np.nanmean(s1[1]):.1f} dB')
    assert -30 < np.nanmean(s1[0]) < 5, f'SAR dB out of reasonable bounds in {zone}.'
    print(f'  --> [{zone}] P0 ALL PASS\n')
for z in ['pune', 'satara', 'vidarbha']:
    p0_check(z)


## Cell 4: Single-Source Auto-Label Engine Import (`data.autolabel`)

Imports the multi-modal evidence fusion engine directly from `data.autolabel` (single source of truth).

In [ ]:
# ============ CELL 4: IMPORT FUSION ENGINE ============
from data.autolabel import (
    load_evidence, build_evidence_masks, fuse, report,
    CLASS_NAMES, IGNORE, CFG
)

print('Successfully imported data.autolabel (Single Source of Truth):')
print('  Classes:', CLASS_NAMES)
print('  Ignore Index:', IGNORE)
print('  Fusion Parameters:', CFG)

## Cell 5: Zone A Percentile Tuning Sweep (STOP & TUNE)

**Architectural Gate (FINAL_ARCH_3.md line 6868 & 7565):**
- Sweep `ae_pct_lo` in `[45, 50, 55, 60, 65]` on Zone A (Pune).
- **Target for Zone A:** `uncertain` in **15–30%**, `change` in **8–20%**.
- **STRICT GATE REQUIREMENT:** If no percentile hits the target window, the script **fails loudly with a RuntimeError**. It does NOT silently fall back to an arbitrary default.
- **FREEZE RULE:** The selected percentile is frozen and applied identically to Zones B & C.

In [ ]:
# ============ CELL 5: ZONE A PERCENTILE SWEEP ============
zone = 'pune'
o1, n1, s1 = load_zone(zone, 'T1')
o2, n2, s2 = load_zone(zone, 'T2')

def calc_idx(img, b_a, b_b):
    return (img[IDX[b_a]] - img[IDX[b_b]]) / (img[IDX[b_a]] + img[IDX[b_b]] + EPS)

ndvi1, ndvi2 = calc_idx(o1, 'B8', 'B4'),  calc_idx(o2, 'B8', 'B4')
ndbi1, ndbi2 = calc_idx(o1, 'B11','B8'),  calc_idx(o2, 'B11','B8')
mnd1,  mnd2  = calc_idx(o1, 'B3', 'B11'), calc_idx(o2, 'B3', 'B11')

ev = load_evidence(zone, DRIVE_RAW, load_merged)
ae = load_merged(f'{zone}_alphaearth_change', DRIVE_RAW)[0].astype(np.float32) / 10000.0
masks = build_evidence_masks(ev, ndvi1, ndvi2, ndbi1, ndbi2, mnd1, mnd2)

print('--- SWEEPING ae_pct_lo ON ZONE A (PUNE) ---')
sweep_results = []
for lo in [45, 50, 55, 60, 65]:
    c = {**CFG, 'ae_pct_lo': lo}
    lab, _ = fuse(masks, ae, c)
    u_pct = 100.0 * float((lab == IGNORE).mean())
    chg_pct = 100.0 * float(((lab > 0) & (lab != IGNORE)).mean())
    is_valid = (15.0 <= u_pct <= 30.0 and 8.0 <= chg_pct <= 20.0)
    status = 'VALID (PASS)' if is_valid else 'OUT OF BOUNDS'
    print(f'ae_pct_lo={lo:2d}: uncertain={u_pct:5.1f}%  change={chg_pct:5.1f}%  [{status}]')
    sweep_results.append((lo, u_pct, chg_pct))

# Filter candidate percentiles meeting the scientific target window
valid_los = [lo for lo, u, c in sweep_results if 15.0 <= u <= 30.0 and 8.0 <= c <= 20.0]

if not valid_los:
    raise RuntimeError(
        'CRITICAL GATE FAILURE: No ae_pct_lo meets the Zone-A scientific gate targets '\
        '(Target: uncertain 15-30%, change 8-20%). '\
        'Do NOT proceed to Cell 6 until the cause is investigated and resolved.'
    )

SELECTED_LO = valid_los[0]
print(f'\n>>> ZONE A GATE PASSED!')
print(f'>>> FROZEN PERCENTILE CHOSEN: ae_pct_lo = {SELECTED_LO}')
print('>>> This exact threshold will be applied to Zone B and Zone C.')

## Cell 6: Multi-Zone Auto-Label Generation (Using Frozen Threshold)

Executes fusion across `pune`, `satara`, and `vidarbha` using the **frozen `SELECTED_LO`** and writes:
- `pune_autolabel_full.npz`
- `satara_autolabel_full.npz`
- `vidarbha_autolabel_full.npz`
- `autolabel_report.json`

In [ ]:
# ============ CELL 6: FUSE ALL THREE ZONES — ULTRA-MEMORY-SAFE ============
import gc

SELECTED_LO = 65
FINAL_CFG = {**CFG, 'ae_pct_lo': SELECTED_LO}
reports = []

def calc_idx_local(bA, bB):
    return (bA - bB) / (bA + bB + 1e-6)

for z in ['pune', 'satara', 'vidarbha']:
    print(f'\n===== Fusing evidence layers for [{z}] =====')
    show_ram(f'{z} start')

    # --- PROCESS T1 ---
    o1_raw = load_merged(f'{z}_T1_optical', DRIVE_RAW).astype(np.float32)
    b3 = o1_raw[IDX['B3']] / S2_SCALE
    b4 = o1_raw[IDX['B4']] / S2_SCALE
    b8 = o1_raw[IDX['B8']] / S2_SCALE
    b11 = o1_raw[IDX['B11']] / S2_SCALE
    
    n1_idx = calc_idx_local(b8, b4)
    b1_idx = calc_idx_local(b11, b8)
    m1_idx = calc_idx_local(b3, b11)
    
    del o1_raw, b3, b4, b8, b11
    gc.collect()
    show_ram(f'{z} after T1 indices')

    # --- PROCESS T2 ---
    o2_raw = load_merged(f'{z}_T2_optical', DRIVE_RAW).astype(np.float32)
    b3 = o2_raw[IDX['B3']] / S2_SCALE
    b4 = o2_raw[IDX['B4']] / S2_SCALE
    b8 = o2_raw[IDX['B8']] / S2_SCALE
    b11 = o2_raw[IDX['B11']] / S2_SCALE
    
    n2_idx = calc_idx_local(b8, b4)
    b2_idx = calc_idx_local(b11, b8)
    m2_idx = calc_idx_local(b3, b11)
    
    del o2_raw, b3, b4, b8, b11
    gc.collect()
    show_ram(f'{z} after T2 indices')

    # --- PROCESS EVIDENCE & AE ---
    ev_z = load_evidence(z, DRIVE_RAW, load_merged)
    ae_z = load_merged(f'{z}_alphaearth_change', DRIVE_RAW)[0].astype(np.float32) / 10000.0
    show_ram(f'{z} after evidence + AE')

    # --- FUSE ---
    masks_z = build_evidence_masks(
        ev_z,
        n1_idx, n2_idx,
        b1_idx, b2_idx,
        m1_idx, m2_idx,
        FINAL_CFG
    )
    
    lab_z, conf_z = fuse(masks_z, ae_z, FINAL_CFG)
    rep = report(lab_z, z)
    reports.append(rep)

    assert 8.0 <= rep['uncertain'] <= 40.0, f'[{z}] uncertain {rep["uncertain"]}% outside allowed gate [8%, 40%].'
    if z == 'pune':
        assert rep['construction'] >= 1.0, f'Zone A construction ({rep["construction"]}%) < 1.0%. Check OB thresholds.'

    # Save immediately
    np.savez_compressed(
        LOCAL / f'{z}_autolabel_full.npz',
        label=lab_z,
        conf=(conf_z * 100).astype(np.uint8)
    )
    print(f'[{z}] saved successfully.')

    # Clean up everything for this zone
    del (
        n1_idx, n2_idx,
        b1_idx, b2_idx,
        m1_idx, m2_idx,
        ev_z, ae_z,
        masks_z, lab_z, conf_z
    )
    gc.collect()
    show_ram(f'{z} freed')

json.dump(reports, open(LOCAL / 'autolabel_report.json', 'w'), indent=2)
print('\nAll three zones fused successfully.')
print('Saved autolabel maps and autolabel_report.json.')


## Cell 7: Spatial AOI Split & 128x128 Tiling with Auto-Labels

Produces:
- `pune_train.npy`, `pune_test.npy`
- `satara_train.npy`, `satara_test.npy`
- `vidarbha_test.npy` (100% held-out test)
- Corresponding `_label.npy`, `_conf.npy`, and `_meta.json` files.

In [ ]:
# ============ CELL 7: SPLIT & STREAMING TILING (BOUNDED RAM) ============
import gc

def show_ram_local(tag=''):
    try:
        show_ram(tag)
    except NameError:
        pass

def split_masks(H, W, zone):
    """
    Returns (train_mask, test_mask) with a BUFFER_PX gap between them.
    """
    tr = np.zeros((H, W), bool); te = np.zeros((H, W), bool)
    if zone == 'pune':                 # West trains, East tests
        cut = W // 2
        tr[:, :cut - BUFFER_PX // 2] = True
        te[:,  cut + BUFFER_PX // 2:] = True
    elif zone == 'satara':             # North trains, South tests
        cut = H // 2
        tr[:cut - BUFFER_PX // 2, :] = True
        te[ cut + BUFFER_PX // 2:, :] = True
    else:                              # Vidarbha = 100% held out
        te[:] = True
    return tr, te


def quantize_i16_inplace(x, scale=10000.0):
    """Scale float32 scene and convert to safe int16 with in-place operations."""
    np.multiply(x, scale, out=x)
    np.clip(x, -32768, 32767, out=x)
    return x.astype(np.int16)


def open_npy_memmap(path, shape, dtype):
    """Create a disk-backed .npy array without materializing the full output in RAM."""
    return np.lib.format.open_memmap(path, mode='w+', dtype=dtype, shape=shape)


def tile_zone_v32(zone: str, out_dir: Path):
    print(f'\n--- Tiling Zone [{zone}] ---')
    show_ram_local(f'{zone} start')
    
    # Process T1: load -> derive -> quantize -> free float32 immediately
    o1, n1, s1 = load_zone(zone, 'T1')
    x1 = derive_17ch(o1, n1, s1)
    del o1, n1, s1
    x1_i16 = quantize_i16_inplace(x1)
    del x1
    gc.collect()
    
    # Process T2: load -> derive -> quantize -> free float32 immediately
    o2, n2, s2 = load_zone(zone, 'T2')
    x2 = derive_17ch(o2, n2, s2)
    del o2, n2, s2
    x2_i16 = quantize_i16_inplace(x2)
    del x2
    gc.collect()
    show_ram_local(f'{zone} after int16 conversion')
    
    full_npz = np.load(out_dir / f'{zone}_autolabel_full.npz')
    lab, conf_u8 = full_npz['label'], full_npz['conf']
    _, H, W = x1_i16.shape
    tr_m, te_m = split_masks(H, W, zone)

    def collect_and_write(mask, stride, tag):
        # Scan bounding boxes (integers only)
        locs, M = [], []
        for r in range(0, H - PATCH + 1, stride):
            for cc in range(0, W - PATCH + 1, stride):
                if not mask[r:r+PATCH, cc:cc+PATCH].all():
                    continue
                pl = lab[r:r+PATCH, cc:cc+PATCH]
                if (pl == IGNORE).mean() > 0.80:
                    continue
                # check finite slice
                p1_slice = x1_i16[:, r:r+PATCH, cc:cc+PATCH]
                p2_slice = x2_i16[:, r:r+PATCH, cc:cc+PATCH]
                if not (np.isfinite(p1_slice).all() and np.isfinite(p2_slice).all()):
                    continue
                locs.append((r, cc))
                M.append({
                    'zone': zone, 'split': tag, 'row': int(r), 'col': int(cc),
                    'q_mean': float(p1_slice[16].mean() / 10000.0),
                    'chg_frac': float(((pl > 0) & (pl != IGNORE)).mean()),
                    'unc_frac': float((pl == IGNORE).mean())
                })
        
        N = len(locs)
        if N == 0:
            np.save(out_dir / f'{zone}_{tag}.npy', np.zeros((0, 2, 17, PATCH, PATCH), dtype=np.int16))
            np.save(out_dir / f'{zone}_{tag}_label.npy', np.zeros((0, PATCH, PATCH), dtype=np.uint8))
            np.save(out_dir / f'{zone}_{tag}_conf.npy', np.zeros((0, PATCH, PATCH), dtype=np.uint8))
        else:
            # Stream directly to disk via open_memmap -- zero multi-GB RAM allocation
            arr = open_npy_memmap(out_dir / f'{zone}_{tag}.npy', (N, 2, 17, PATCH, PATCH), np.int16)
            labels = open_npy_memmap(out_dir / f'{zone}_{tag}_label.npy', (N, PATCH, PATCH), np.uint8)
            confs = open_npy_memmap(out_dir / f'{zone}_{tag}_conf.npy', (N, PATCH, PATCH), np.uint8)
            
            for i, (r, cc) in enumerate(locs):
                arr[i, 0] = x1_i16[:, r:r+PATCH, cc:cc+PATCH]
                arr[i, 1] = x2_i16[:, r:r+PATCH, cc:cc+PATCH]
                labels[i] = lab[r:r+PATCH, cc:cc+PATCH]
                confs[i]  = conf_u8[r:r+PATCH, cc:cc+PATCH]
                
            arr.flush()
            labels.flush()
            confs.flush()
            del arr, labels, confs
            gc.collect()
            
        json.dump(M, open(out_dir / f'{zone}_{tag}_meta.json', 'w'))
        mean_chg = 100 * np.mean([x['chg_frac'] for x in M]) if M else 0.0
        print(f'  {zone}/{tag}: patches={N}  mean change={mean_chg:.1f}%')

    for tag, mask, stride in [('train', tr_m, STRIDE_TR), ('test', te_m, STRIDE_TE)]:
        collect_and_write(mask, stride, tag)

    del x1_i16, x2_i16, lab, conf_u8, tr_m, te_m
    gc.collect()
    show_ram_local(f'{zone} cleanup')

# Stream each zone independently directly to disk
for z in ['pune', 'satara', 'vidarbha']:
    tile_zone_v32(z, LOCAL)
show_ram_local('Cell 7 End')



## Cell 8: Normalization Statistics (TRAIN AOI ONLY — Zero Leakage)

Computes channel-wise mean and std from **TRAIN AOI ONLY** (Pune West + Satara North). Zero test pixels are included.

In [ ]:
# ============ CELL 8: NORM STATS (IN-PLACE CHUNKED STREAMING) ============
import gc
ARRAY_SCALE = 10000.0
SAR_EXTRA_SCALE = 100.0
CHUNK_SIZE  = 128  # 128 patches = ~272 MiB float32 working buffer

def compute_norm_stats(zones, data_dir: Path):
    n = 0
    sm = np.zeros(17, dtype=np.float64)
    sq = np.zeros(17, dtype=np.float64)
    
    print('--- Computing Channel Normalization Statistics (Train AOI Only) ---')
    for z in zones:
        path = data_dir / f'{z}_train.npy'
        if not path.exists():
            continue
        marr = np.load(path, mmap_mode='r')
        total = marr.shape[0]
        print(f'  Streaming {z}/train: {total} patches (mmap)...')
        
        for i in range(0, total, CHUNK_SIZE):
            j = min(i + CHUNK_SIZE, total)
            # [B, 2, 17, H, W] stored int16 -> float32 model units (~272 MiB working buffer)
            chunk = marr[i:j].astype(np.float32, copy=True)
            chunk /= ARRAY_SCALE
            
            # Restore SAR dB scale BEFORE computing statistics so normalization
            # matches the model input scale restored by MHPatches loader (Part 16.3).
            for ch in SAR_CH:
                chunk[:, :, ch, :, :] *= SAR_EXTRA_SCALE
            
            # [B, 2, 17, H, W] -> sum directly over axes (0, 1, 3, 4) down to 17 channels
            sm += chunk.sum(axis=(0, 1, 3, 4), dtype=np.float64)
            # Reuse the same buffer for squares in-place
            np.square(chunk, out=chunk)
            sq += chunk.sum(axis=(0, 1, 3, 4), dtype=np.float64)
            n  += (chunk.shape[0] * chunk.shape[1] * chunk.shape[3] * chunk.shape[4])
            del chunk
        del marr
        gc.collect()

    assert n > 0, 'No training pixels found!'
    mu    = sm / n
    sigma = np.sqrt(np.maximum(sq / n - mu ** 2, 1e-12))

    # Architecture: indices and quality mask remain unnormalized
    for c in (11, 12, 16):  # NDVI, NDBI, q
        mu[c] = 0.0
        sigma[c] = 1.0

    stats = {
        'mean': mu.tolist(),
        'std': sigma.tolist(),
        'array_scale': ARRAY_SCALE,
        'sar_channels': SAR_CH,
        'sar_extra_scale': SAR_EXTRA_SCALE,
        'n_pixels': int(n),
        'source': 'TRAIN AOI patches only (pune west + satara north)'
    }
    json.dump(stats, open(data_dir / 'norm_stats_trainonly.json', 'w'), indent=2)
    
    channel_labels = S2_BANDS + ['NDVI','NDBI','VV','VH','CR','q']
    for i, b in enumerate(channel_labels):
        print(f'  ch{i:2d} {b:5s}  mu={mu[i]:+.4f}  sd={sigma[i]:.4f}')
    return stats

stats = compute_norm_stats(['pune', 'satara'], LOCAL)
show_ram('Cell 8 End')


## Cell 9: Independent 220-Patch Annotation Pool & High-Performance QGIS Export

Fulfills the exact protocol and allocation matrix specified in `FINAL_ARCH_3.md` Part 18, Part 22.5, and Part 26.2 (lines 6510–6528):
- `MH-VAL`: 30 (15 Pune train + 15 Satara train)
- `MH-ADAPT`: 30 (15 Pune train + 15 Satara train)
- `MH-TEST dry A`: 40 (Pune test)
- `MH-TEST dry B`: 40 (Satara test)
- `MH-TEST blind`: 20 (10 Pune test + 10 Satara test, with `blind=True` — raw imagery only, NO auto-labels and NO AlphaEarth to eliminate anchoring bias per Part 22.5)
- `MH-TEST monsoon`: 30 (Pune test, loaded strictly from the **monsoon image pairs** `pune_monsoon_T1` and `pune_monsoon_T2`, stratified independently by cloud quality $q$, with `is_monsoon=True` scaling $q255$, and `blind=True` to carry Hypothesis H2)
- `MH-TEST Vidarbha`: 30 (Vidarbha test for Hypothesis H3)
- **TOTAL: EXACTLY 220 PATCHES**

**High-Performance Architecture:** Full-scene rasters are preloaded once into a memory cache, eliminating hundreds of redundant GeoTIFF file reads.

In [ ]:
# ============ CELL 9: 220-PATCH POOL & HIGH-PERFORMANCE QGIS EXPORT ============
from data.export_qgis import export_patch

# AlphaEarth cache for sampling routines (memoized: loads each zone exactly once)
ae_sampling_cache = {}
def get_ae_chg(zone: str):
    if zone not in ae_sampling_cache:
        ae_sampling_cache[zone] = load_merged(f'{zone}_alphaearth_change', DRIVE_RAW).astype(np.float32)[0] / 10000.0
    return ae_sampling_cache[zone]

def build_pool_sample(zone: str, split: str, n_target: int, seed: int = 0, exclude_set: set = None):
    """Stratified sampling across AlphaEarth change distribution."""
    if exclude_set is None: exclude_set = set()
    chg = get_ae_chg(zone)
    meta_path = LOCAL / f'{zone}_{split}_meta.json'
    if not meta_path.exists(): return []
    meta = json.load(open(meta_path))
    
    valid_items = [m for m in meta if (zone, m['row'], m['col']) not in exclude_set]
    if not valid_items: return []
        
    score = np.array([chg[m['row']:m['row']+PATCH, m['col']:m['col']+PATCH].mean() for m in valid_items])
    q60, q85, q95 = np.percentile(score, [60, 85, 95])
    strata = {
        'high':      (score >= q95,                       0.40),
        'moderate':  ((score >= q85) & (score < q95),      0.30),
        'ambiguous': ((score >= q60) & (score < q85),      0.20),
        'stable':    (score < q60,                         0.10)
    }

    rng = np.random.default_rng(seed)
    chosen = []
    for name, (mask, frac) in strata.items():
        idx = np.flatnonzero(mask)
        k = min(int(round(n_target * frac)), len(idx))
        if len(idx) > 0 and k > 0:
            for i in rng.choice(idx, size=k, replace=False):
                item = valid_items[int(i)]
                chosen.append({**item, 'stratum': name, 'ae_score': float(score[i])})
                exclude_set.add((zone, item['row'], item['col']))
                
    remaining_needed = n_target - len(chosen)
    if remaining_needed > 0:
        available = [m for m in valid_items if (zone, m['row'], m['col']) not in exclude_set]
        if available:
            fill_idx = rng.choice(len(available), size=min(remaining_needed, len(available)), replace=False)
            for i in fill_idx:
                item = available[int(i)]
                chosen.append({**item, 'stratum': 'fill', 'ae_score': float(chg[item['row']:item['row']+PATCH, item['col']:item['col']+PATCH].mean())})
                exclude_set.add((zone, item['row'], item['col']))
                
    print(f'  {zone}/{split} -> selected {len(chosen)} / {n_target} target patches.')
    return chosen


def build_monsoon_pool_sample(n_target: int = 30, seed: int = 50, exclude_set: set = None):
    """
    Independent monsoon sampling across Pune test half, stratified strictly by 
    monsoon cloud quality q (cs_cdf) rather than dry AlphaEarth change.
    Derives candidate patches directly from monsoon raster across the Pune Test AOI grid,
    completely independent of dry auto-labels and dry AlphaEarth distributions.
    Fulfills Hypothesis H2 testing protocol (FINAL_ARCH_3.md Part 18 & Part 26.2).
    """
    if exclude_set is None: exclude_set = set()
    
    # Load Pune monsoon T2 optical to derive spatial grid & cloud quality q
    o2_m = load_merged('pune_monsoon_T2_optical', DRIVE_RAW).astype(np.float32)
    _, H, W = o2_m.shape
    q_field = np.clip(o2_m[11] / 255.0, 0.0, 1.0)
    
    # Candidate grid locations covering Pune Test AOI (East half, past the buffer gap)
    cut = W // 2
    c_start = cut + BUFFER_PX // 2
    
    valid_items = []
    for r in range(0, H - PATCH + 1, STRIDE_TE):
        for cc in range(c_start, W - PATCH + 1, STRIDE_TE):
            if ('pune', r, cc) in exclude_set:
                continue
            p_opt = o2_m[:11, r:r+PATCH, cc:cc+PATCH]
            if not np.isfinite(p_opt).all():
                continue
            valid_items.append({'zone': 'pune', 'split': 'test', 'row': int(r), 'col': int(cc)})
            
    if not valid_items: return []
    
    from collections import Counter
    scores = np.array([q_field[m['row']:m['row']+PATCH, m['col']:m['col']+PATCH].mean() for m in valid_items])
    
    # Exact Architecture-defined H2 strata: q <= 0.4 / 0.4 < q <= 0.8 / q > 0.8 (FINAL_ARCH_3.md line 366)
    assert n_target == 30, f"Expected n_target=30 for MH-TEST monsoon, got {n_target}"
    target_per_stratum = n_target // 3  # Exactly 10 patches per stratum
    
    rng = np.random.default_rng(seed)
    chosen = []
    strata_definitions = [
        ('heavy_cloud',    scores <= 0.40),
        ('moderate_cloud', (scores > 0.40) & (scores <= 0.80)),
        ('clear_sky',      scores > 0.80),
    ]
    
    for name, mask in strata_definitions:
        idx = np.flatnonzero(mask)
        assert len(idx) >= target_per_stratum, (
            f"H2 Protocol Error: Stratum '{name}' has only {len(idx)} eligible candidate patches in "
            f"Pune Test AOI, but exactly {target_per_stratum} are required by FINAL_ARCH_3.md line 366."
        )
        for i in rng.choice(idx, size=target_per_stratum, replace=False):
            item = valid_items[int(i)]
            chosen.append({**item, 'stratum': name, 'cloud_q_mean': float(scores[i])})
            exclude_set.add(('pune', item['row'], item['col']))
            
    counts = Counter(x['stratum'] for x in chosen)
    assert counts == {'heavy_cloud': 10, 'moderate_cloud': 10, 'clear_sky': 10}, (
        f"Monsoon stratification failure: expected exactly 10 per stratum, got {counts}"
    )
    print(f'  pune/monsoon -> selected exactly 30 patches: {dict(counts)}')
    return chosen

# Construct the exact 220-patch evaluation allocation (Part 26.2)
claimed_patches = set()
pool = {}

print('--- Sampling 220-Patch Verification Sets ---')
# 1. MH-VAL (30: 15 Pune train + 15 Satara train)
pool['MH_VAL'] = (
    build_pool_sample('pune', 'train', 15, seed=10, exclude_set=claimed_patches) +
    build_pool_sample('satara', 'train', 15, seed=11, exclude_set=claimed_patches)
)

# 2. MH-ADAPT (30: 15 Pune train + 15 Satara train)
pool['MH_ADAPT'] = (
    build_pool_sample('pune', 'train', 15, seed=20, exclude_set=claimed_patches) +
    build_pool_sample('satara', 'train', 15, seed=21, exclude_set=claimed_patches)
)

# 3. MH-TEST dry A (40: Pune test)
pool['MH_TEST_DRY_A'] = build_pool_sample('pune', 'test', 40, seed=30, exclude_set=claimed_patches)

# 4. MH-TEST dry B (40: Satara test)
pool['MH_TEST_DRY_B'] = build_pool_sample('satara', 'test', 40, seed=31, exclude_set=claimed_patches)

# 5. MH-TEST blind (20: 10 Pune test + 10 Satara test, drawn from scratch)
pool['MH_TEST_BLIND'] = (
    build_pool_sample('pune', 'test', 10, seed=40, exclude_set=claimed_patches) +
    build_pool_sample('satara', 'test', 10, seed=41, exclude_set=claimed_patches)
)

# 6. MH-TEST monsoon (30: Pune test, drawn independently stratified by monsoon cloud quality q)
pool['MH_TEST_MONSOON'] = build_monsoon_pool_sample(30, seed=50, exclude_set=claimed_patches)

# 7. MH-TEST Vidarbha (30: Vidarbha test for Hypothesis H3)
pool['MH_TEST_VIDARBHA'] = build_pool_sample('vidarbha', 'test', 30, seed=60, exclude_set=claimed_patches)

total_patches = sum(len(v) for v in pool.values())
print(f'\nTotal Patches Sampled: {total_patches} (Target: 220)')
assert total_patches == 220, f'Annotation pool total is {total_patches}, expected exactly 220.'
json.dump(pool, open(LOCAL / 'annotation_pool.json', 'w'), indent=2)

# Clear sampling cache before full-scene export stage
ae_sampling_cache.clear()
gc.collect()

# ============ HIGH-PERFORMANCE ZONE-BY-ZONE EXPORT ============
# Process one geographic zone at a time to prevent concurrent multi-zone RAM exhaustion.
# Peak memory depends on scene dimensions and runtime state; measures RSS rather than assuming a fixed bound.
import gc

print('\n--- Setting Up Georeferencing & Export Paths ---')
zone_georef = {}
for z in ['pune', 'satara', 'vidarbha']:
    src_tif = sorted(DRIVE_RAW.glob(f'{z}_T1_optical*.tif'))[0]
    with rasterio.open(src_tif) as src:
        zone_georef[z] = (src.transform, src.crs)

mon_tif = sorted(DRIVE_RAW.glob('pune_monsoon_T1_optical*.tif'))[0]
with rasterio.open(mon_tif) as src:
    zone_georef['pune_monsoon'] = (src.transform, src.crs)

# Organize patch tasks by zone so each zone is loaded into memory exactly once
tasks_by_zone = {'pune': [], 'satara': [], 'vidarbha': [], 'pune_monsoon': []}
for split_name, items in pool.items():
    is_blind = ('BLIND' in split_name or 'MONSOON' in split_name)
    is_monsoon = ('MONSOON' in split_name)
    out_dir = LOCAL / 'qgis' / split_name
    out_dir.mkdir(parents=True, exist_ok=True)
    for it in items:
        geo_key = 'pune_monsoon' if is_monsoon else it['zone']
        tasks_by_zone[geo_key].append({
            'zone': it['zone'], 'row': it['row'], 'col': it['col'],
            'out_dir': out_dir, 'blind': is_blind, 'split': split_name
        })

print('\n--- Exporting QGIS Packages by Streaming One Zone at a Time ---')
for geo_key in ['pune', 'satara', 'vidarbha', 'pune_monsoon']:
    tasks = tasks_by_zone[geo_key]
    if not tasks:
        continue
    print(f'\nLoading [{geo_key}] into memory ({len(tasks)} patches to export)...')
    show_ram(f'{geo_key} start')
    base_tf, crs = zone_georef[geo_key]
    
    if geo_key == 'pune_monsoon':
        o1_m, n1_m, s1_m = load_zone('pune_monsoon', 'T1')
        o2_m, n2_m, s2_m = load_zone('pune_monsoon', 'T2')
        x1 = derive_17ch(o1_m, n1_m, s1_m, is_monsoon=True)
        x2 = derive_17ch(o2_m, n2_m, s2_m, is_monsoon=True)
        lab, ae = None, None
        del o1_m, n1_m, s1_m, o2_m, n2_m, s2_m
    else:
        o1, n1, s1 = load_zone(geo_key, 'T1')
        o2, n2, s2 = load_zone(geo_key, 'T2')
        x1, x2 = derive_17ch(o1, n1, s1), derive_17ch(o2, n2, s2)
        npz = np.load(LOCAL / f'{geo_key}_autolabel_full.npz')
        lab = npz['label']
        ae = get_ae_chg(geo_key)
        del o1, n1, s1, o2, n2, s2
        
    for t in tqdm(tasks, desc=f'Exporting {geo_key} patches'):
        export_patch(
            t['zone'], t['row'], t['col'],
            x1, x2, lab, ae,
            base_tf, crs, t['out_dir'], blind=t['blind']
        )
        
    # Free memory for this zone before loading next
    del x1, x2, lab, ae
    gc.collect()
    show_ram(f'{geo_key} freed')

print('\nQGIS Verification Stubs successfully generated in /content/proc/qgis/!')


## Cell 10: Drive Archive & Kaggle Staging Preparation

Packs `/content/proc` into a single archive on Google Drive:
- `geonexus_v3_processed.tar`

### Licensing & Staging Notes:
- Uses license `'other'` with compliant attribution to Copernicus Sentinel open access data and upstream evidence sources (Google Open Buildings Temporal V1 — CC-BY-4.0 or ODbL-1.0, Google Dynamic World CC-BY-4.0, Hansen GFC CC-BY-4.0, JRC Global Surface Water).
- Staging workflow options:
  - **Option A (Direct Colab Staging):** Authenticate and run `!kaggle datasets create -p /content/proc --dir-mode zip`.
  - **Option B (Local Staging):** Extract `geonexus_v3_processed.tar` locally to `data/processed/` and run `python data/stage_kaggle.py --user sumit07125 --dataset geonexus-mh-v3 --new`.

In [ ]:
# ============ CELL 10: ARCHIVE & PREPARE KAGGLE STAGING ============
print('Creating tar archive in local scratch...')
!cd /content && tar -cf /content/geonexus_v3_processed.tar -C /content proc

print(f'Copying archive to Google Drive ({DRIVE_PROC})...')
!cp /content/geonexus_v3_processed.tar "{DRIVE_PROC}/geonexus_v3_processed.tar"
print('Successfully archived processed dataset to Drive!')

# Dataset metadata for Kaggle staging with compliant mixed-source license
meta = {
    'title': 'Geo-Nexus Maharashtra CD v3.2 Training Data',
    'id': 'sumit07125/geonexus-mh-v3',
    'licenses': [{'name': 'other'}],
    'description': (
        'Multi-modal optical (Sentinel-2) and SAR (Sentinel-1) bi-temporal change detection '
        'training arrays and evidence auto-labels for Maharashtra, India. '
        'Derived from Copernicus Sentinel open access data, with upstream multi-modal '
        'evidence layers from Google Dynamic World (CC-BY-4.0), Google Open Buildings Temporal V1 '
        '(CC-BY-4.0 or ODbL-1.0), Hansen Global Forest Change (CC-BY-4.0), and JRC Global '
        'Surface Water (EC open data / Copernicus programme acknowledgement).'
    )
}
json.dump(meta, open(LOCAL / 'dataset-metadata.json', 'w'), indent=2)
print('dataset-metadata.json generated. Staging preparation complete.')

# Direct Colab Kaggle Upload (Optional if authenticated via kaggle.json in Colab):
# !mkdir -p ~/.kaggle && cp /content/drive/MyDrive/kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets create -p /content/proc --dir-mode zip